In [35]:
# Imports for the modeling notebook.
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import norm
import biogeme.biogeme as bio
from biogeme.expressions import bioDraws, log, Elem
from biogeme import models
from biogeme.models import ordered_probit, ordered_logit
from biogeme import results as res
import pandas as pd
import biogeme.database as db
import pickle


In [36]:

df = pd.read_csv('final_processed_crash_dataset.csv', low_memory=False)


## Sanity check and preparing the databases

In [37]:
df=df[df['severity']!=-1]
df=df.loc[df['Vehicle'].isin(['E-PMD','Bike','E-bike','Pedestrian'])]

df_dummies=pd.get_dummies(df[['Crossroad','Helmet','Point of impact','Gender','vehicle_type_2','Vehicle','Pavement','Intersection','vehicle_type_3'
                              , 'User category', 'Lighting conditions', 'Cycle facilities', 'Road width', 'Agglomeration', 'Age category',
                              'Accident location', 'Surface condition', 'Maneuver', 'Maneuver_2','Gender_2', 'Pedestrian localisation', 'Pedestrian action', 'Vehicle_2',
'Max speed', 'Long profile', 'Weather conditions', 'Road type', 'Trip purpose','Reflective jacket', 'plan','Point of impact_2', 'Obstacle','Gender_driver','Helmet_driver','age_driver','Year', 'Maneuver_3','Point of impact_3','Gender_3','Vehicle_3'

]])
df_dummies = df_dummies.astype(int)  # Convert boolean to integers

continuous_vars = [
    'age',
    'Number of passengers',
    'number of involved vehicles',
    'vma',
    'age_2',
    'age_opposite_mean'
]

df_centered = df.copy()
# cellule 3, avant le centrage
df_centered['vma_raw'] = df['vma']
df_centered['Number of passengers raw'] = df['Number of passengers']


# `age_opposite_mean == 0` n'est pas un age : c'est le code des collisions sans
# tiers identifie (2 212 des 2 214 lignes du segment sans tiers). On le traite
# comme manquant AVANT de centrer, sinon ces zeros tirent la moyenne vers le bas
# et decalent toute la variable.
df_centered['age_opposite_mean'] = df_centered['age_opposite_mean'].mask(
    df_centered['age_opposite_mean'] == 0
)

df_centered[continuous_vars] = (
    df_centered[continuous_vars]
    - df_centered[continuous_vars].mean()
)


df_non_dummies = df_centered[['age','severity','Number of passengers','number of involved vehicles','vma','vma_raw','age_2','catu', 'Num_Acc','age_opposite_mean']]




In [38]:
df_non_dummies=pd.concat([df_non_dummies,df_dummies],axis=1)

## Removing useless rows
df_non_dummies=df_non_dummies.dropna(subset=['age','severity'])


df_non_dummies['age_opposite_missing'] = (
    df_non_dummies['age_opposite_mean'].isna().astype(int)
)
df_non_dummies['age_opposite_mean'] = df_non_dummies['age_opposite_mean'].fillna(0.0)

df_non_dummies = df_non_dummies.reset_index(drop=True)

In [39]:
## First model
df_motorized_vehicles=df_non_dummies.loc[(df_non_dummies['vehicle_type_2_Cars']==1) | (df_non_dummies['vehicle_type_2_Large motorized vehicle'] ==1) | (df_non_dummies['vehicle_type_2_Light motorized vehicle']==1) ]
df_motorized_vehicles=df_motorized_vehicles.loc[df_motorized_vehicles['catu'].isin([1,2])]
database_motorized_vehicles = db.Database('database_motorized_vehicles', df_motorized_vehicles)

## Second model
df_mmv=df_non_dummies.loc[(df_non_dummies['vehicle_type_2_Micromobility vehicle']==1) ]
df_mmv=df_mmv.loc[df_mmv['catu'].isin([1,2])]

df_mmv = df_mmv.copy()  # age du tiers manquant : conserve, via age_opposite_missing

database_mmv= db.Database('database_mmv',df_mmv)


## Third model
num_acc_values = df_non_dummies.loc[df_non_dummies['vehicle_type_2_Pedestrian'] == 1, 'Num_Acc']
df_pedestrian = df_non_dummies[df_non_dummies['Num_Acc'].isin(num_acc_values)]
df_pedestrian = df_pedestrian.copy()  # idem : plus aucune ligne supprimee ici

database_pedestrian= db.Database('database_pedestrian',df_pedestrian)

## Fourth model
df_sv=df_non_dummies.loc[df_non_dummies['vehicle_type_2_No other vehicle']==1]
df_sv=df_sv.loc[df_sv['catu'].isin([1,2])]
database_sv= db.Database('database_sv',df_sv)


## Variables and Betas

In [40]:
import re
import biogeme.database as db
from biogeme.expressions import Beta, Variable


def normalize(name: str) -> str:
    clean = name.lower()
    clean = re.sub(r'[^a-z0-9]+', '_', clean)
    clean = re.sub(r'_+', '_', clean)
    clean = clean.strip('_')
    if re.match(r'^[0-9]', clean):
        clean = "var_" + clean
    return clean

base_suffixes = ['','_I', '_F', '_mean', '_I_mean', '_F_mean','_sd','_I_std','_F_std']
modes = ['', '_epmd', '_bike', '_ebike']  

for col in df_non_dummies.columns:
    clean_name = normalize(col)

    # Skip si nettoyage donne v_injuryde
    if clean_name == "":
        print(f"Skip (empty after cleaning): {col}")
        continue

    if clean_name not in locals():
        exec(f"{clean_name} = Variable({repr(col)})")
    # Pour chaque suffixe de base et chaque mode, créer un Beta si non existant
    for suf in base_suffixes:
        for mode in modes:
            beta_var_name = f"beta_{clean_name}{suf}{mode}"
            beta_label = beta_var_name

            if '_std' in suf or suf == '_std':
                init_value = 1
            else:
                init_value = 0

            if not beta_var_name.isidentifier():
                print(f"Skip beta (invalid identifier): {beta_var_name}")
                continue

            if beta_var_name in locals():
                continue
            exec(f"{beta_var_name} = Beta({repr(beta_label)}, {init_value}, None, None, 0)")

constant_I = Beta('constant_I', 0, None, None, 0)
constant_F = Beta('constant_F', 0, None, None, 0)


harmed = (severity > 1) + 1
availability_harmed = {1: 1, 2: 1}


# Save the results in appropriate folders

In [41]:

import os
from contextlib import contextmanager
from pathlib import Path

RESULTS_ROOT = Path('results')
RESULTS_DIRECTORIES = {
    'motorized_vehicles': RESULTS_ROOT / 'model_motorized_vehicles',
    'mmv': RESULTS_ROOT / 'model_mmv',
    'pedestrian': RESULTS_ROOT / 'model_pedestrian',
    'single_vehicle': RESULTS_ROOT / 'model_single_vehicle',
}
for directory in RESULTS_DIRECTORIES.values():
    directory.mkdir(parents=True, exist_ok=True)


@contextmanager
def _in_results_directory(segment):

    previous = Path.cwd()
    os.chdir(RESULTS_DIRECTORIES[segment])
    try:
        yield
    finally:
        os.chdir(previous)


def save_results(results, segment, model_name):

    results.data.modelName = model_name
    with _in_results_directory(segment):
        for extension in ('html', 'pickle'):
            Path(f'{model_name}.{extension}').unlink(missing_ok=True)
        results.write_html(True)
        results.write_pickle()
    return results


def estimate_in(segment, the_biogeme, **kwargs):
    """Estimate a model, writing its HTML/pickle output into `segment`'s directory."""
    with _in_results_directory(segment):
        return the_biogeme.estimate(**kwargs)


def validate_in(segment, the_biogeme, estimation_results, validation_data):
    """Out-of-sample validation, writing its output into `segment`'s directory."""
    with _in_results_directory(segment):
        return the_biogeme.validate(estimation_results, validation_data)

## Model for car crashes

### Baseline (constants only)

We first estimate the constants-only model. Its log-likelihood is the
denominator used for the rho-square statistics reported below and for the
likelihood-ratio test (see the LR-test section).


In [42]:
## Constant model

availability = {
    1: 1, 
    2: 1, 
    3: 1   
}

U={1:0,2:constant_I,3:constant_F}

model_name = 'InitialModel_motorized_vehicles'


logprob = models.loglogit(U, availability, severity)

# Créez l'objet Biogeme
model_cst_car = bio.BIOGEME(database_motorized_vehicles, logprob)
model_cst_car.modelName = model_name

# Estimation


results_constant_car = estimate_in('motorized_vehicles', model_cst_car)



### Full mixed-logit specification


In [43]:

def any_of(*terms):
    total = terms[0]
    for term in terms[1:]:
        total = total + term
    return total > 0


v_no_injury = 0

v_injury = (  beta_gender_female_I * gender_female  + constant_I +beta_age_I*age
      + beta_user_category_passenger_I * user_category_passenger
     + beta_number_of_involved_vehicles_I * number_of_involved_vehicles
      + beta_point_of_impact_back_I * point_of_impact_back
      + beta_vehicle_type_2_light_motorized_vehicle_I * any_of(vehicle_type_2_light_motorized_vehicle,
                                                             vehicle_type_3_light_motorized_vehicle)
    # + beta_maneuver_2_overtaking_I * any_of(maneuver_2_overtaking, maneuver_3_overtaking)
      + beta_maneuver_without_change_of_direction_I * maneuver_without_change_of_direction
    #  + user_category_passenger * (beta_gender_driver_female_I * gender_driver_female)
     + beta_intersection_no_intersection_I * (intersection_no_intersection)
# + beta_maneuver_2_overtaking_I * any_of(maneuver_2_overtaking, maneuver_3_overtaking)
)

v_fatality = (  constant_F
     
        + beta_age_F * age

        + beta_vma_F * vma * intersection_no_intersection
        + beta_vehicle_type_2_large_motorized_vehicle_F * any_of(vehicle_type_2_large_motorized_vehicle,
                                                                vehicle_type_3_large_motorized_vehicle)
                                                           
        + beta_lighting_conditions_night_with_street_lightings_on_F * (lighting_conditions_night_with_street_lightings_on + lighting_conditions_night_without_street_lightings)
        + beta_maneuver_2_turning_right_F * any_of(maneuver_2_turning_right, maneuver_3_turning_right)
        + beta_accident_location_on_cycle_facility_F* accident_location_on_cycle_facility
       # + beta_lighting_conditions_night_without_street_lightings_F* lighting_conditions_night_without_street_lightings
   #    + user_category_passenger * (beta_gender_driver_female_I * gender_driver_female)
    # + beta_intersection_no_intersection_F  * intersection_no_intersection
    
)

utility_motorized_vehicles = {
    1: v_no_injury,
    2: v_injury,
    3: v_fatality
}


In [44]:
# Random-parameters model
sigma_I = Beta('sigma I', 0, None, None, 0)

X1 = bioDraws('X1', 'NORMAL')


# Adding the error component to the utilities
v_no_injury_rp=0
v_injury_rp = utility_motorized_vehicles[2] 
v_fatality_rp = utility_motorized_vehicles[3]

utility_motorized_vehicles_mixed={1:v_no_injury_rp,2:v_injury_rp,3:v_fatality_rp}

prob = models.logit(utility_motorized_vehicles_mixed,availability,severity)


logprob = log((prob))



# Create the Biogeme object
model_car  = bio.BIOGEME(database_motorized_vehicles,logprob)
model_car.modelName = "logit_car_crashes"

# Estimate the parameters. 
results_ml_motorized_vehicles = estimate_in('motorized_vehicles', model_car)


In [45]:
results_ml_motorized_vehicles.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_accident_location_on_cycle_facility_F,-1.000931,0.514956,-1.943722,5.192895e-02
beta_age_F,0.054315,0.009249,5.872457,4.293837e-09
beta_age_I,0.015784,0.004564,3.458632,5.429272e-04
beta_gender_female_I,0.629694,0.135199,4.657524,3.200356e-06
beta_intersection_no_intersection_I,0.314262,0.131498,2.389860,1.685478e-02
beta_lighting_conditions_night_with_street_lightings_on_F,0.882966,0.288059,3.065227,2.175048e-03
beta_maneuver_2_turning_right_F,1.229583,0.312264,3.937638,8.228771e-05
beta_maneuver_without_change_of_direction_I,0.414924,0.115763,3.584241,3.380603e-04
beta_number_of_involved_vehicles_I,-0.963034,0.204516,-4.708845,2.491250e-06
beta_point_of_impact_back_I,-0.388049,0.151633,-2.559138,1.049321e-02


### Ordered-probit counterpart




In [46]:
# --- Car crashes: ordered-probit counterpart of the MNL above -----------------

car_index_terms = {
    'gender_female': gender_female,
    'age': age,
    'user_category_passenger': user_category_passenger,
   # 'impact_back_bike': point_of_impact_back * (vehicle_bike + vehicle_e_bike),
   # 'impact_back_epmd': point_of_impact_back * vehicle_e_pmd,
    'light_motorized_vehicle': vehicle_type_2_light_motorized_vehicle,
    'large_motorized_vehicle': vehicle_type_2_large_motorized_vehicle,
#    'overtaking': maneuver_2_overtaking,
#    'no_change_of_direction_bike': maneuver_without_change_of_direction * vehicle_bike,
    'female_driver_passenger': user_category_passenger * gender_driver_female,
#    'at_intersection': (intersection_no_intersection == 0),
    'vma_no_intersection': vma * intersection_no_intersection,
   # 'night_street_lightings_on': lighting_conditions_night_with_street_lightings_on,
#    'night_no_street_lightings': lighting_conditions_night_without_street_lightings,
    'turning_right': maneuver_2_turning_right,
#    'on_cycle_facility': accident_location_on_cycle_facility,
}

index_car = None
for name, term in car_index_terms.items():
    contribution = Beta(f'beta_{name}_car_probit', 0, None, None, 0) * term
    index_car = contribution if index_car is None else index_car + contribution

model_name = 'ordered_probit_motorized_vehicles'
the_proba = ordered_probit(
    continuous_value=index_car,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=Beta('tau_1_car_probit', 1, None, None, 0),
)
logprob = log(Elem(the_proba, severity))
model_car_probit = bio.BIOGEME(database_motorized_vehicles, logprob)
model_car_probit.modelName = model_name
results_car_probit = estimate_in('motorized_vehicles', model_car_probit)

print(f'MNL            : LL={results_ml_motorized_vehicles.data.logLike:9.3f}  '
      f'K={results_ml_motorized_vehicles.data.nparam:3d}  '
      f'AIC={results_ml_motorized_vehicles.data.akaike:8.2f}')
print(f'ordered probit : LL={results_car_probit.data.logLike:9.3f}  '
      f'K={results_car_probit.data.nparam:3d}  '
      f'AIC={results_car_probit.data.akaike:8.2f}')
results_car_probit.get_estimated_parameters().round(4)


MNL            : LL=-1447.007  K= 16  AIC= 2926.01
ordered probit : LL=-1498.691  K= 10  AIC= 3017.38


,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age_car_probit,0.0085,0.0018,4.6192,0.0000
beta_female_driver_passenger_car_probit,-0.4523,0.2268,-1.9941,0.0461
beta_gender_female_car_probit,0.1752,0.0515,3.3997,0.0007
beta_large_motorized_vehicle_car_probit,0.6118,0.1619,3.7784,0.0002
beta_light_motorized_vehicle_car_probit,-1.0884,0.0583,-18.6828,0.0000
beta_turning_right_car_probit,0.1347,0.0787,1.7115,0.0870
beta_user_category_passenger_car_probit,-0.8751,0.1515,-5.7750,0.0000
beta_vma_no_intersection_car_probit,0.0115,0.0046,2.5095,0.0121
tau_1_car_probit,-2.1225,0.0418,-50.7776,0.0000
tau_1_car_probit_diff_2,4.7161,0.0676,69.7567,0.0000


## MMV

### Baseline (constants only)


In [47]:
v_no_injurym=0
v_injurym = constant_I


U={1:v_no_injurym,2:v_injurym}

In [48]:
model_name = 'InitialModel_mmv'

# La disponibilite, pas les utilites : les deux alternatives sont toujours
# offertes. L'ancienne version passait {1: 0, 2: constant_I}, ce qui rendait
# l'alternative 1 indisponible -- d'ou l'avertissement "chosen alternative
# is not available" sur toutes les lignes sans blessure -- et faisait d'un
# parametre estime une disponibilite.
availability1 = {1: 1, 2: 1}

logprob_2 = models.loglogit(U, availability_harmed, harmed)

# Créez l'objet Biogeme
model_cst_mmv = bio.BIOGEME(database_mmv, logprob_2)
model_cst_mmv.modelName = model_name

# Estimation


results_constant_mmv = estimate_in('mmv', model_cst_mmv)



### Full logit specification


In [49]:
v_no_injury = 0

v_injury = (
      beta_gender_female_I                  * gender_female
    + constant_I
    + beta_gender_2_female_I               * (gender_2_female * (age_opposite_missing == 0))
    + beta_point_of_impact_back_I_bike     * point_of_impact_back * (vehicle_bike + vehicle_e_bike)
    #    + beta_point_of_impact_back_I_epmd     * point_of_impact_back * (vehicle_e_pmd)
#+ beta_vehicle_2_e_pmd_I*any_of(vehicle_2_e_pmd, vehicle_3_e_pmd)*(age_opposite_missing==0)
    + beta_surface_condition_wet_I         * surface_condition_wet
    + beta_age_I                           * age
     + beta_maneuver_swerving_I           * maneuver_swerving
    + beta_maneuver_turning_left_I         * maneuver_turning_left
    + beta_age_2_I                          * age_opposite_mean
    + beta_age_opposite_missing_I           * age_opposite_missing
)

# Dictionnaire des fonctions d'utilité
utility_mmv = {
    1: v_no_injury,
    2: v_injury,
}


In [50]:
availability1={1:1,2:1}



logprob_2 = models.loglogit(utility_mmv, availability_harmed, harmed)
model_mmv = bio.BIOGEME(database_mmv, logprob_2)
model_mmv.modelName = 'logit_mmv'
results_logit_mmv = estimate_in('mmv', model_mmv)
results_logit_mmv.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age_2_I,-0.026044,0.004776,-5.452572,4.964640e-08
beta_age_I,0.033870,0.005205,6.507280,7.652390e-11
beta_age_opposite_missing_I,8.346715,0.164674,50.686181,0.000000e+00
beta_gender_2_female_I,-1.066366,0.144015,-7.404534,1.316725e-13
beta_gender_female_I,1.093919,0.168711,6.483970,8.933987e-11
beta_maneuver_swerving_I,-0.475088,0.190064,-2.499618,1.243272e-02
beta_maneuver_turning_left_I,-1.310990,0.320989,-4.084218,4.422557e-05
beta_point_of_impact_back_I_bike,-1.117479,0.221624,-5.042228,4.601417e-07
beta_surface_condition_wet_I,-0.565295,0.230921,-2.448003,1.436503e-02
constant_I,0.808105,0.097425,8.294630,0.000000e+00


### Binary probit counterpart

With two outcomes the ordered logit *is* the binary logit estimated above, so
the ordered/unordered comparison has nothing to arbitrate here: only the link
function can be varied. `ordered_probit` over two discrete values is exactly a
binary probit. Its coefficients carry the opposite sign, because the ordered
parameterisation models P(severity = 1) = Phi(tau - V), and they are on the
normal rather than the logistic scale -- what is comparable is the
log-likelihood, hence the AIC.

The constant is dropped from the index: in the ordered parameterisation the
threshold `tau` plays that role, and keeping both would not be identified.


In [51]:
# --- MMV: binary probit counterpart of the binary logit above -----------------

index_mmv = (
      Beta('beta_gender_female_mmv_probit', 0, None, None, 0) * gender_female
    + Beta('beta_gender_2_female_mmv_probit', 0, None, None, 0)
      * (gender_2_female + gender_3_female)
    + Beta('beta_impact_back_bike_mmv_probit', 0, None, None, 0)
      * point_of_impact_back * (vehicle_bike + vehicle_e_bike)
    + Beta('beta_impact_back_epmd_mmv_probit', 0, None, None, 0)
      * point_of_impact_back * vehicle_e_pmd
    + Beta('beta_surface_wet_mmv_probit', 0, None, None, 0) * surface_condition_wet
    + Beta('beta_age_mmv_probit', 0, None, None, 0) * age
    + Beta('beta_swerving_mmv_probit', 0, None, None, 0) * maneuver_swerving
    + Beta('beta_turning_left_mmv_probit', 0, None, None, 0) * maneuver_turning_left
    + Beta('beta_age_opposite_mean_mmv_probit', 0, None, None, 0) * age_opposite_mean
  #  + Beta('beta_vehicle_2_e_pmd_mmv_probit', 0, None, None, 0)
  #    * (vehicle_2_e_pmd + vehicle_3_e_pmd)
   #       + beta_age_opposite_missing_I           * age_opposite_missing

)

model_name = 'binary_probit_mmv'
the_proba = ordered_probit(
    continuous_value=index_mmv,
    list_of_discrete_values=[1, 2],
    tau_parameter=Beta('tau_1_mmv_probit', 1, None, None, 0),
)
logprob = log(Elem(the_proba, harmed))
model_mmv_probit = bio.BIOGEME(database_mmv, logprob)
model_mmv_probit.modelName = model_name
results_mmv_probit = estimate_in('mmv', model_mmv_probit)

print(f'binary logit  : LL={results_logit_mmv.data.logLike:9.3f}  '
      f'K={results_logit_mmv.data.nparam:3d}  '
      f'AIC={results_logit_mmv.data.akaike:8.2f}')
print(f'binary probit : LL={results_mmv_probit.data.logLike:9.3f}  '
      f'K={results_mmv_probit.data.nparam:3d}  AIC={results_mmv_probit.data.akaike:8.2f}')
results_mmv_probit.get_estimated_parameters().round(4)


binary logit  : LL= -662.368  K= 10  AIC= 1344.74
binary probit : LL= -688.605  K= 10  AIC= 1397.21


,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age_mmv_probit,0.0203,0.0029,6.9202,0.0000
beta_age_opposite_mean_mmv_probit,-0.0141,0.0029,-4.9086,0.0000
beta_gender_2_female_mmv_probit,-0.7279,0.0854,-8.5248,0.0000
beta_gender_female_mmv_probit,0.6384,0.0947,6.7449,0.0000
beta_impact_back_bike_mmv_probit,-0.6847,0.1293,-5.2944,0.0000
beta_impact_back_epmd_mmv_probit,0.7608,0.3673,2.0710,0.0384
beta_surface_wet_mmv_probit,-0.3218,0.1341,-2.3998,0.0164
beta_swerving_mmv_probit,-0.3306,0.1142,-2.8938,0.0038
beta_turning_left_mmv_probit,-0.7420,0.1807,-4.1071,0.0000
tau_1_mmv_probit,-0.5762,0.0565,-10.2049,0.0000


## Pedestrian

### Baseline (constants only)

Ordered-probit baseline with a flat utility (`continuous_value=0`) and the
single threshold `tau_1`.


In [52]:
model_name = 'ordered_probit_pedestrian_init'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=0,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_cst_pedes = bio.BIOGEME(database_pedestrian, logprob)
model_cst_pedes.modelName = model_name
results_pedes_cst = estimate_in('pedestrian', model_cst_pedes)


### Full ordered-probit specification


In [53]:
utility_pedestrian = (
      beta_gender_female                     * gender_female
    + beta_age                               * age
    + beta_intersection_no_intersection      * (intersection_no_intersection)
    + beta_age_opposite_mean                             * age_opposite_mean
    + beta_age_opposite_missing                          * age_opposite_missing
    + beta_gender_2_female                   * gender_2_female
    + beta_user_category_pedestrian          * user_category_pedestrian
    # « Au moins un des tiers tourne » : la somme brute donnerait 2 si le
    # premier et le second tiers tournaient tous les deux, et ignorait le cas
    # ou seul le second tiers tourne.

    + beta_crossroad_traffic_lights          * crossroad_traffic_lights
)


In [54]:
# --- Pedestrian: ordered-probit counterpart of the ordered logit above --------
# Same index (`utility_pedestrian`) and same threshold structure; only the link
# function changes. The two estimations are independent, so the Beta objects can
# be shared: each `estimate()` starts from the initial values again.

model_name = 'ordered_probit_pedestrian'

tau_1_pedes_probit = Beta('tau_1_pedes_probit', 1, None, None, 0)
the_proba = ordered_probit(
    continuous_value=utility_pedestrian,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1_pedes_probit,
)
logprob = log(Elem(the_proba, severity))
model_pedes_probit = bio.BIOGEME(database_pedestrian, logprob)
model_pedes_probit.modelName = model_name
results_pedes_probit = estimate_in('pedestrian', model_pedes_probit)


results_pedes_probit.get_estimated_parameters().round(4)

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age,0.0146,0.0016,9.0996,0.0000
beta_age_opposite_mean,-0.0109,0.0015,-7.2933,0.0000
beta_age_opposite_missing,1.2872,0.4877,2.6393,0.0083
beta_crossroad_traffic_lights,0.1994,0.0779,2.5593,0.0105
beta_gender_2_female,-0.5870,0.0653,-8.9956,0.0000
beta_gender_female,0.5116,0.0670,7.6336,0.0000
beta_intersection_no_intersection,0.2220,0.0694,3.1959,0.0014
beta_user_category_pedestrian,1.3222,0.0728,18.1609,0.0000
tau_1_pedes_probit,0.4929,0.0717,6.8749,0.0000
tau_1_pedes_probit_diff_2,4.1839,0.1666,25.1170,0.0000


### MNL counterpart of the same specification

The same nine variables, but one coefficient per alternative instead of a single
latent index. The ordered probit forces a variable to push severity in one
direction only; the MNL lets it raise the odds of injury and lower those of
death, at the cost of twice the coefficients.

The interaction `pedestrian x turning` never occurs among the fatalities, so its
coefficient on that alternative is not identified. The cell prints the counts.


In [55]:

from biogeme.expressions import MonteCarlo

xi_ped = bioDraws('xi_ped', 'NORMAL')
sigma_injury_ped = Beta('sigma_injury_ped', 1, None, None, 0)
beta_maneuver_2_turning = Beta('beta_maneuver_2_turning', 0, None, None, 0)
v_injury_ped_mnl = (Beta('asc_injury_ped', 0, None, None, 0)
   # + sigma_injury_ped * xi_ped
    + beta_gender_female_I                     * gender_female
    + beta_age_I                               * age
    + beta_intersection_no_intersection_I      * (intersection_no_intersection == 0)
    + beta_age_opposite_mean_I                 * age_opposite_mean
    + beta_age_opposite_missing_I              * age_opposite_missing
    + beta_gender_2_female_I                   * gender_2_female
    + beta_user_category_pedestrian_I          * user_category_pedestrian
    + beta_crossroad_traffic_lights_I          * crossroad_traffic_lights
      
)


utility_pedestrian_mnl = {1: 0, 2: v_injury_ped_mnl}

model_name = 'logit_pedestrian'

logprob = models.loglogit(utility_pedestrian_mnl, availability_harmed, harmed)

model_pedestrian_mnl = bio.BIOGEME(database_pedestrian, logprob)
model_pedestrian_mnl.modelName = model_name
results_pedestrian_mnl = estimate_in('pedestrian', model_pedestrian_mnl)
results_pedestrian_mnl.get_estimated_parameters().round(4)

,Value,Rob. Std err,Rob. t-test,Rob. p-value
asc_injury_ped,-0.4553,0.1084,-4.2009,0.0000
beta_age_I,0.0266,0.0028,9.4136,0.0000
beta_age_opposite_mean_I,-0.0184,0.0027,-6.9144,0.0000
beta_age_opposite_missing_I,2.1181,0.8289,2.5552,0.0106
beta_crossroad_traffic_lights_I,0.3356,0.1495,2.2457,0.0247
beta_gender_2_female_I,-1.0949,0.1139,-9.6112,0.0000
beta_gender_female_I,0.9958,0.1163,8.5628,0.0000
beta_intersection_no_intersection_I,-0.3567,0.1305,-2.7326,0.0063
beta_user_category_pedestrian_I,2.2874,0.1240,18.4541,0.0000


In [56]:

model_name = 'logit_pedestrian_cst'

logprob = models.loglogit(
    {1: 0, 2: Beta('asc_harmed_ped_cst', 0, None, None, 0)},
    availability_harmed,
    harmed,
)
model_cst_pedes_mnl = bio.BIOGEME(database_pedestrian, logprob)
model_cst_pedes_mnl.modelName = model_name
results_pedes_cst_mnl = estimate_in('pedestrian', model_cst_pedes_mnl)

print(f'LL(c) binary logit  : {results_pedes_cst_mnl.data.logLike:10.3f}  '
      f'(2 modalites de `harmed`)')
print(f'LL(c) ordered probit: {results_pedes_cst.data.logLike:10.3f}  '
      f'(3 niveaux de `severity`)')
results_pedes_cst_mnl.get_estimated_parameters().round(4)


LL(c) binary logit  :  -1901.711  (2 modalites de `harmed`)
LL(c) ordered probit:  -1957.378  (3 niveaux de `severity`)


,Value,Rob. Std err,Rob. t-test,Rob. p-value
asc_harmed_ped_cst,0.312,0.0383,8.1426,0.0


## Single-vehicle

### Baseline (constants only)

Ordered-probit baseline (`continuous_value=0`).


In [57]:
model_name = 'ordered_probit_s_init'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=0,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_cst_solo = bio.BIOGEME(database_sv, logprob)
model_cst_solo.modelName = model_name
results_cst_solo = estimate_in('single_vehicle', model_cst_solo)
results_cst_solo.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
tau_1,-2.115844,0.064947,-32.578097,0.0
tau_1_diff_2,4.395907,0.099121,44.348801,0.0


### Full ordered-probit specification


In [58]:
utility_sv = (
   beta_age * age +
    beta_user_category_passenger * user_category_passenger +
    beta_long_profile_slope *long_profile_slope 
     + beta_helmet_yes_ebike*helmet_yes*(vehicle_e_bike)
  #  + beta_number_of_passengers*user_category_driver*(number_of_passengers>0)
)


In [59]:
model_name = 'ordered_probit_sinv'

tau_1 = Beta('tau_1', 1, None, None, 0)
   
# :math:`\tau_2 = \tau_2 + \delta_2`
the_proba = ordered_probit(
    continuous_value=utility_sv,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1,
)

the_chosen_proba = Elem(the_proba, severity)

logprob = log((the_chosen_proba))
model_solo_2 = bio.BIOGEME(database_sv, logprob)
model_solo_2.modelName = model_name
results_solo_2 = estimate_in('single_vehicle', model_solo_2)
results_solo_2.get_estimated_parameters()

,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age,0.011139,0.003141,3.546718,3.900612e-04
beta_helmet_yes_ebike,-0.186187,0.128277,-1.451450,1.466547e-01
beta_long_profile_slope,0.303959,0.152739,1.990062,4.658406e-02
beta_user_category_passenger,-1.359961,0.227456,-5.979001,2.245101e-09
tau_1,-2.290107,0.086301,-26.536223,0.000000e+00
tau_1_diff_2,4.672406,0.123737,37.760716,0.000000e+00


### Ordered-logit counterpart

The same index with a logistic error term.


In [60]:
del_name = 'ordered_logit_sinv'

tau_1_solo_logit = Beta('tau_1_solo_logit', 1, None, None, 0)
the_proba = ordered_logit(
    continuous_value=utility_sv,
    list_of_discrete_values=[1, 2, 3],
    tau_parameter=tau_1_solo_logit,
)
logprob = log(Elem(the_proba, severity))
model_solo_logit = bio.BIOGEME(database_sv, logprob)
model_solo_logit.modelName = model_name
results_solo_logit = estimate_in('single_vehicle', model_solo_logit)

for label, results in (('ordered probit', results_solo_2),
                       ('ordered logit ', results_solo_logit)):
    print(f'{label} : LL={results.data.logLike:9.3f}  K={results.data.nparam:3d}  '
          f'AIC={results.data.akaike:8.2f}  BIC={results.data.bayesian:8.2f}')
results_solo_logit.get_estimated_parameters().round(4)

ordered probit : LL= -289.842  K=  6  AIC=  591.68  BIC=  625.89
ordered logit  : LL= -288.703  K=  6  AIC=  589.41  BIC=  623.62


,Value,Rob. Std err,Rob. t-test,Rob. p-value
beta_age,0.0272,0.0078,3.5041,0.0005
beta_helmet_yes_ebike,-0.4584,0.3971,-1.1544,0.2483
beta_long_profile_slope,0.7003,0.3694,1.8960,0.0580
beta_user_category_passenger,-2.9654,0.4072,-7.2832,0.0000
tau_1_solo_logit,-4.5354,0.2261,-20.0612,0.0000
tau_1_solo_logit_diff_2,9.2668,0.3186,29.0885,0.0000


### MNL counterpart of the same specification




In [61]:
beta_helmet_yes_ebike_I = Beta('beta_helmet_yes_ebike_I', 0, None, None, 0)
beta_helmet_yes_ebike_F = Beta('beta_helmet_yes_ebike_F', 0, None, None, 0)

v_injury_sv = (Beta('asc_injury_sv', 0, None, None, 0)
    + beta_age_I * age
    + beta_user_category_passenger_I * user_category_passenger
    + beta_vehicle_e_pmd*vehicle_e_pmd

   # + beta_number_of_passengers_I * user_category_driver * number_of_passengers  # ajouté
)
v_fatality_sv = (Beta('asc_fatality_sv', 0, None, None, 0)
 +   beta_age_F * age +
    beta_long_profile_slope_F *long_profile_slope 
)


utility_sv_mnl = {1: 0, 2: v_injury_sv, 3: v_fatality_sv}



model_name = 'mnl_sinv'
logprob = models.loglogit(utility_sv_mnl, availability, severity)
model_solo_mnl = bio.BIOGEME(database_sv, logprob)
model_solo_mnl.modelName = model_name
results_solo_mnl = estimate_in('single_vehicle', model_solo_mnl)



In [62]:

model_name = 'mnl_sinv_cst'

logprob = models.loglogit(
    {1: 0,
     2: Beta('asc_injury_sv_cst', 0, None, None, 0),
     3: Beta('asc_fatality_sv_cst', 0, None, None, 0)},
    availability,
    severity,
)
model_cst_solo_mnl = bio.BIOGEME(database_sv, logprob)
model_cst_solo_mnl.modelName = model_name
results_cst_solo_mnl = estimate_in('single_vehicle', model_cst_solo_mnl)

print(f'LL(c) MNL           : {results_cst_solo_mnl.data.logLike:10.3f}')
print(f'LL(c) ordered probit: {results_cst_solo.data.logLike:10.3f}')
results_cst_solo_mnl.get_estimated_parameters().round(4)


LL(c) MNL           :   -328.598
LL(c) ordered probit:   -328.598


,Value,Rob. Std err,Rob. t-test,Rob. p-value
asc_fatality_sv_cst,-0.4189,0.2575,-1.6269,0.1038
asc_injury_sv_cst,4.0350,0.1636,24.6610,0.0000


In [63]:
results_solo_mnl.get_estimated_parameters().round(4)

,Value,Rob. Std err,Rob. t-test,Rob. p-value
asc_fatality_sv,-0.5719,0.3214,-1.7792,0.0752
asc_injury_sv,4.8369,0.2450,19.7399,0.0000
beta_age_F,0.0705,0.0147,4.7845,0.0000
beta_age_I,0.0267,0.0127,2.0984,0.0359
beta_long_profile_slope_F,1.0120,0.4496,2.2508,0.0244
beta_user_category_passenger_I,-2.6405,0.3609,-7.3166,0.0000
beta_vehicle_e_pmd,-0.7133,0.2824,-2.5260,0.0115


## AME


In [ ]:

from biogeme.expressions import Derive, Variable, exp

# Au-dela de ce nombre de valeurs distinctes, la variable est traitee comme
# continue (derivee) ; en deca, comme un comptage (variation discrete de +1).
DISCRETE_MAX_LEVELS = 10







KEY_VARIABLES = {
    'Car crashes': [
        'age', 'number of involved vehicles',
        ('Second-party heavy vehicle',
         ['vehicle_type_2_Large motorized vehicle',
          'vehicle_type_3_Large motorized vehicle']),
        ('Second-party light vehicle',
         ['vehicle_type_2_Light motorized vehicle',
          'vehicle_type_3_Light motorized vehicle']),
        ('Crash at night',
         ['Lighting conditions_Night with street lightings on',
          'Lighting conditions_Night without street lightings']),
        'Accident location_On cycle facility',
        'Intersection_No intersection',
        ('Second-party turning right',
         ['Maneuver_2_Turning right', 'Maneuver_3_Turning right']),
        'User category_Passenger', 'Gender_Female',
    ],
    'MMV': [
        'age', 
        'age_opposite_mean', 'Gender_Female', 'Surface condition_Wet',
        'Maneuver_Swerving', 'Maneuver_Turning left', 
        'Point of impact_Back',
    ],
    'Pedestrian': [
        'age', 'age_opposite_mean', 'Gender_Female', 'User category_Pedestrian',
        'Crossroad_Traffic lights', 'Intersection_No intersection',
    ],
    'Single-vehicle': [
        'age', 'User category_Passenger', 'Long profile_Slope',
        'Number of passengers',
    ],
}


def _model_variables(the_biogeme, exclude=('severity',)):
    """Noms des colonnes qui apparaissent dans la formule du modele."""
    found, stack, seen = set(), [the_biogeme.log_like], set()
    while stack:
        node = stack.pop()
        if id(node) in seen:
            continue
        seen.add(id(node))
        if isinstance(node, Variable):
            found.add(node.name)
        stack.extend(node.get_children())
    return sorted(found - set(exclude))


def _outcome_simulators(the_biogeme, data, outcomes, choice_column='severity'):
    simulators = {}
    for outcome in outcomes:
        frame = data.copy()
        frame[choice_column] = outcome
        simulator = bio.BIOGEME(db.Database('marginal_effects', frame),
                                {'probability': exp(the_biogeme.log_like)},
                                number_of_draws=the_biogeme.number_of_draws,
                                generate_html=False, generate_pickle=False)
        simulator.modelName = f'{the_biogeme.modelName}_simulation'
        simulators[outcome] = simulator
    return simulators


def _derivative_simulators(the_biogeme, data, outcomes, column,
                           choice_column='severity'):

    simulators = {}
    for outcome in outcomes:
        frame = data.copy()
        frame[choice_column] = outcome
        simulator = bio.BIOGEME(
            db.Database('marginal_effects', frame),
            {'derivative': Derive(exp(the_biogeme.log_like), column)},
            number_of_draws=the_biogeme.number_of_draws,
            generate_html=False, generate_pickle=False)
        simulator.modelName = f'{the_biogeme.modelName}_derivative'
        simulators[outcome] = simulator
    return simulators


def marginal_effects(the_biogeme, results, variables=None, outcome_labels=None,
                     choice_column='severity'):

    data = the_biogeme.database.data.copy()
    betas = results.get_beta_values()
    outcomes = sorted(int(value) for value in data[choice_column].unique())
    labels = outcome_labels or {outcome: f'P(severity = {outcome})'
                                for outcome in outcomes}

    available = _model_variables(the_biogeme, exclude=(choice_column,))

    def normalise(entry):
        if isinstance(entry, str):
            return entry, [entry]
        label, columns = entry
        return label, list(columns)

    if variables is None:
        selection = [(name, [name]) for name in available]
    else:
        selection = []
        for entry in variables:
            label, columns = normalise(entry)
            kept = [column for column in columns if column in available]
            missing = [column for column in columns if column not in available]
            if missing:
                print(f'  [{label}] hors du modele, ignoree : {", ".join(missing)}')
            if kept:
                selection.append((label, kept))

    records = []
    for label, columns in selection:
        derivative_column = None
        column = data[columns[0]]
        values = set(pd.concat([data[name] for name in columns]).dropna().unique())
        if values <= {0, 1}:
      
            scale, kind = 1.0, 'dummy (0 to 1)'
            frame_low = data.assign(**{name: 0.0 for name in columns})
            frame_high = data.assign(**{name: 1.0 if name == columns[0] else 0.0
                                        for name in columns})
        elif column.nunique() <= DISCRETE_MAX_LEVELS:
            scale, kind = 1.0, 'per unit (+1)'
            frame_low = data
            frame_high = data.assign(**{columns[0]: column + 1})
        else:
            if len(columns) > 1:
                continue
            if column.nunique() < 2:
                continue                       
            scale, kind, derivative_column = 1.0, 'per unit', columns[0]

        if derivative_column is not None:
            simulators = _derivative_simulators(the_biogeme, data, outcomes,
                                                derivative_column, choice_column)

            def effect(outcome):
                simulated = simulators[outcome].simulate(betas)['derivative']
                return 100 * float(simulated.mean())
        else:
            low = _outcome_simulators(the_biogeme, frame_low, outcomes,
                                      choice_column)
            high = _outcome_simulators(the_biogeme, frame_high, outcomes,
                                       choice_column)

            def effect(outcome):
                difference = (high[outcome].simulate(betas)['probability']
                              - low[outcome].simulate(betas)['probability'])
                return 100 * float(difference.mean()) / scale

        record = {'Variable': label, 'Variation': kind}
        for outcome in outcomes:
            record[labels[outcome]] = effect(outcome)
        records.append(record)

    table = pd.DataFrame.from_records(records).set_index('Variable')
    point_columns = [labels[outcome] for outcome in outcomes]
    ordering = table[point_columns].abs().max(axis=1)
    table = table.loc[ordering.sort_values(ascending=False).index]


    simulators = _outcome_simulators(the_biogeme, data, outcomes, choice_column)
    baseline = {'Variable': BASELINE_LABEL, 'Variation': 'baseline'}
    for outcome in outcomes:
        baseline[labels[outcome]] = 100 * float(
            simulators[outcome].simulate(betas)['probability'].mean())
    baseline = pd.DataFrame.from_records([baseline]).set_index('Variable')
    return pd.concat([baseline, table])


SEVERITY_LABELS = {1: 'No injury', 2: 'Injury', 3: 'Fatality'}
MAIN_MODELS = {
    'Car crashes': (results_ml_motorized_vehicles, model_car, 'carcrashes'),
    'MMV': (results_logit_mmv, model_mmv, 'mmv'),
    'Pedestrian': (results_pedestrian_mnl, model_pedestrian_mnl, 'pedestrian'),
    'Single-vehicle': (results_solo_mnl, model_solo_mnl, 'single_vehicle'),
}
marginal_effects_tables = {}
for label, (results, the_biogeme, segment) in MAIN_MODELS.items():
    print(f'=== {label} : average marginal effects (percentage points) ===')
    try:
        table = marginal_effects(the_biogeme, results,
                                 variables=KEY_VARIABLES.get(label),
                                 outcome_labels=SEVERITY_LABELS)
    except Exception as error:
        print(f'  calcul impossible : {type(error).__name__}: {error}\n')
        continue
    marginal_effects_tables[label] = table
    print(table.round(2).to_string())
    print()



=== Car crashes : average marginal effects (percentage points) ===
                                          Variation  No injury  Injury  Fatality
Variable                                                                        
Baseline predicted probabilities           baseline       3.59   95.71      0.69
User category_Passenger              dummy (0 to 1)      13.78  -16.44      2.65
Second-party light vehicle           dummy (0 to 1)      10.53  -12.76      2.22
Second-party heavy vehicle           dummy (0 to 1)      -0.58   -6.92      7.50
number of involved vehicles           per unit (+1)       4.30   -5.13      0.82
Gender_Female                        dummy (0 to 1)      -1.76    2.10     -0.34
Second-party turning right           dummy (0 to 1)      -0.08   -1.12      1.20
Intersection_No intersection         dummy (0 to 1)      -0.94    0.91      0.03
Crash at night                       dummy (0 to 1)      -0.04   -0.65      0.69
Accident location_On cycle facility  dummy

In [ ]:
def get_results(file_path):
    """Load a Biogeme `bioResults` object back from its pickle file."""
    with open(file_path, 'rb') as file:
        data = pickle.load(file)
    return res.bioResults(data)




## Out-of sample validation of the models

In [ ]:
# Create DataFrames for each year and without each year
years = [2019, 2020, 2021, 2022, 2023]

# Classe pour contenir les données de validation
class ValidationData:
    def __init__(self, estimation, validation):
        self.estimation = estimation
        self.validation = validation

def create_validation_data(df):
    validation_data=[]
    validation_data.append(ValidationData(df[df['Year'].isin([2019, 2020, 2021, 2022])], df[df['Year'] == 2023]))
    df_lyon = df[df['Agglomeration_MÉTROPOLE DE LYON'] == 1]
    df_paris = df[df['Agglomeration_MÉTROPOLE DU GRAND PARIS']==1]
    validation_data.append(ValidationData(df_paris, df_lyon))
    return validation_data



# Create validation data for each dataset
validationData_car = create_validation_data(df_motorized_vehicles)
validationData_mmv= create_validation_data(df_mmv)
validationData_sv_2 = create_validation_data(df_sv)
validationData_pedestrian = create_validation_data(df_pedestrian)

In [ ]:
# Validate the model with the validation data for car
validation_results_car = model_car.validate(results_ml_motorized_vehicles, validationData_car)
validation_results_car_cst = model_cst_car.validate(results_ml_motorized_vehicles, validationData_car)


# Initialize variables to store log-likelihoods

loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (car)
for i, slide in enumerate(validation_results_car):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_car += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on car (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (cars)
for i, slide in enumerate(validation_results_car_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_car += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on car (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (cars)
for i in range(len(validation_results_car)):
    validation_loglike = validation_results_car[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_car_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on car (slide {i+1}): {rho_square}')





Log likelihood for 2129 validation data on car (slide 1): -343.2020030288181
Log likelihood for 1361 validation data on car (slide 2): -227.15160582984856
Log likelihood for 2129 validation data on car (constant model, slide 1): -400.53378892971614
Log likelihood for 1361 validation data on car (constant model, slide 2): -249.8457174826326
Rho-square for the validation data on car (slide 1): 0.14313845045157558
Rho-square for the validation data on car (slide 2): 0.09083250207945459


In [ ]:
# Validate the model with the validation data for mmv
validation_results_mmv = model_mmv.validate(results_logit_mmv, validationData_mmv)
validation_results_mmv_cst = model_cst_mmv.validate(results_constant_mmv, validationData_mmv)


# Initialize variables to store log-likelihoods
loglike_model_mmv = 0
loglike_constant_mmv = 0
loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_mmv):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_mmv += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_mmv_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_mmv += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_mmv)):
    validation_loglike = validation_results_mmv[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_mmv_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on mmv (slide {i+1}): {rho_square}')





Log likelihood for 374 validation data on mmv (slide 1): -191.12330074775184
Log likelihood for 88 validation data on mmv (slide 2): -43.01458640031154
Log likelihood for 374 validation data on mmv (constant model, slide 1): -237.09388505090192
Log likelihood for 88 validation data on mmv (constant model, slide 2): -53.87235429875541
Rho-square for the validation data on mmv (slide 1): 0.1938919019074684
Rho-square for the validation data on mmv (slide 2): 0.20154619265812757


In [ ]:

# Validate the model with the validation data for mmv
validation_results_pedes = model_pedes_probit.validate(results_pedes_probit, validationData_pedestrian)
validation_results_pedes_cst = model_cst_pedes.validate(results_pedes_cst, validationData_pedestrian)


# Initialize variables to store log-likelihoods
loglike_model_pedes= 0
loglike_constant_pedes = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_pedes):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_pedes += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_pedes_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_pedes += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on pedes (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_pedes)):
    validation_loglike = validation_results_pedes[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_pedes_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on pedes (slide {i+1}): {rho_square}')




Log likelihood for 711 validation data on mmv (slide 1): -303.5176586608274
Log likelihood for 223 validation data on mmv (slide 2): -91.04965260534306
Log likelihood for 711 validation data on pedes (constant model, slide 1): -491.2867169242943
Log likelihood for 223 validation data on pedes (constant model, slide 2): -151.9994039479823
Rho-square for the validation data on pedes (slide 1): 0.3821985243952799
Rho-square for the validation data on pedes (slide 2): 0.40098677862906396


In [ ]:


# Validate the model with the validation data for mmv
validation_results_solo_2 = model_solo_2.validate(results_solo_2, validationData_sv_2)
validation_results_solo_cst = model_cst_solo.validate(results_cst_solo, validationData_sv_2)

# Initialize variables to store log-likelihoods
loglike_model_mmv = 0
loglike_constant_mmv = 0
loglike_model_car = 0
loglike_constant_car = 0

# Calculate log-likelihood for the model (mmv)
for i, slide in enumerate(validation_results_solo_2):
    validation_loglike = slide['Loglikelihood'].sum()
    loglike_model_mmv += validation_loglike
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (slide {i+1}): '
        f'{validation_loglike}'
    )

# Calculate log-likelihood for the constant model (mmv)
for i, slide in enumerate(validation_results_solo_cst):
    validation_loglike_cst = slide['Loglikelihood'].sum()
    loglike_constant_mmv += validation_loglike_cst
    print(
        f'Log likelihood for {slide.shape[0]} validation data on mmv (constant model, slide {i+1}): '
        f'{validation_loglike_cst}'
    )

# Calculate rho-square for each slide (mmv)
for i in range(len(validation_results_solo_2)):
    validation_loglike = validation_results_solo_2[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_solo_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)

    # Calculate rho-square for each slide (mmv
for i in range(len(validation_results_solo_2)):
    validation_loglike = validation_results_solo_2[i]['Loglikelihood'].sum()
    validation_loglike_cst = validation_results_solo_cst[i]['Loglikelihood'].sum()
    rho_square = 1 - (validation_loglike / validation_loglike_cst)
    print(f'Rho-square for the validation data on mmv (slide {i+1}): {rho_square}')


Log likelihood for 422 validation data on mmv (slide 1): -72.97280032046456
Log likelihood for 222 validation data on mmv (slide 2): -70.59688434147554
Log likelihood for 422 validation data on mmv (constant model, slide 1): -83.7974453870486
Log likelihood for 222 validation data on mmv (constant model, slide 2): -76.58265527365617
Rho-square for the validation data on mmv (slide 1): 0.12917631338982383
Rho-square for the validation data on mmv (slide 2): 0.07816092182742174
